[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/14_kv_cache.ipynb)

# 🔴 Hard: KV Cache Attention

Implement **multi-head attention with KV caching** for efficient autoregressive generation.

During LLM inference, recomputing all key/value projections at every step is wasteful.
A **KV cache** stores previously computed K and V tensors so only the new token(s) need projection.

### Signature
```python
class KVCacheAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, cache=None) -> tuple[torch.Tensor, tuple]:
        # x: (B, S_new, D) — new tokens
        # cache: None or (K_past, V_past) each (B, num_heads, S_past, d_k)
        # Returns: (output, (K_all, V_all))
```

### Requirements
- Inherit from `nn.Module`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` projections
- When `cache=None` (prefill): apply **causal mask**, return all K/V as cache
- When `cache` provided (decode): concat new K/V with cached, no causal mask needed for single-token decode
- Incremental decode must produce **identical** results to full forward pass

### Key Idea
```
Prefill:  [t0 t1 t2 t3] → full causal attention → cache = (K_{0:3}, V_{0:3})
Decode:   [t4]           → Q=t4, K/V=cache+t4  → cache = (K_{0:4}, V_{0:4})
Decode:   [t5]           → Q=t5, K/V=cache+t5  → cache = (K_{0:5}, V_{0:5})
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn as nn
import math

In [2]:
# ✏️ YOUR IMPLEMENTATION HERE

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        # pass  # Initialize W_q, W_k, W_v, W_o
        self.W_q = nn.Linear(d_model, d_model)  # (d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.W_o = nn.Linear(d_model, d_model)

        assert d_model % num_heads == 0 
        self.nhead = num_heads
    
    # x: (B, S_new, D) — new tokens
    # cache: None or (K_past, V_past) each (B, num_heads, S_past, d_k)

    def forward(self, x, cache=None):
        q, k, v = self.W_q(x), self.W_k(x), self.W_v(x)
        seq_q = q.shape[1]
        bz, seq_k, d_model = k.shape
        dim = d_model//self.nhead
        q = q.reshape(bz, seq_q, self.nhead, dim).transpose(1,2) 
        k = k.reshape(bz, seq_k, self.nhead, dim).transpose(1,2) 
        v = v.reshape(bz, seq_k, self.nhead, dim).transpose(1,2)
        if cache is not None:
            k, v = torch.cat((cache[0], k) , dim=-2), torch.cat((cache[1], v) , dim=-2)
        attn_w = q @ k.transpose(-1,-2) # bz nhead sq sk
        attn_w = attn_w * (dim ** -0.5) # 缩放
        if cache is None: # prefill
            causal_mask = torch.triu(torch.ones(seq_q,seq_k,dtype=bool), diagonal=1)
            attn_w.masked_fill_(causal_mask.unsqueeze(0), float('-inf'))
        attn_w = torch.softmax(attn_w, dim=-1) # softamx
        out = (attn_w @ v).transpose(1,2).reshape(bz, seq_q, -1)
        out = self.W_o(out)
        return out, (k, v)
        # pass

In [3]:
# 🧪 Debug
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 6, 64)

# Full forward
full_out, _ = attn(x)
print("Full output shape:", full_out.shape)  # (1, 6, 64)

# Incremental: prefill 4, decode 1, decode 1
out1, cache = attn(x[:, :4])
print("Cache K shape:", cache[0].shape)  # (1, 4, 4, 16)
out2, cache = attn(x[:, 4:5], cache=cache)
out3, cache = attn(x[:, 5:6], cache=cache)
inc_out = torch.cat([out1, out2, out3], dim=1)
print("Match:", torch.allclose(full_out, inc_out, atol=1e-5))

Full output shape: torch.Size([1, 6, 64])
Cache K shape: torch.Size([1, 4, 4, 16])
Match: True


In [4]:
# ✅ SUBMIT
from torch_judge import check
check('kv_cache')


🧪 Testing: KV Cache Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (no cache) (12.3ms)
  ✅ [2/5] Cache structure (1.3ms)
  ✅ [3/5] Decode step appends to cache (1.3ms)
  ✅ [4/5] Incremental decode matches full forward (1.7ms)
  ✅ [5/5] Gradient flow (39.1ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (55.6ms total)
  Progress saved. Run status() to see your dashboard.

